# 14. Land Value Capture (LVC) Városfejlesztési Szimuláció
## Dinamikus Kétpilléres Közösségi Értéknövekmény-Elvonási Modell

**TDK Kutatási Téma**: A közösségi infrastruktúra-fejlesztések által generált ingatlanpiaci értéktöbblet számszerűsítése és önfinanszírozó visszanyerési mechanizmusainak (LVC) szimulációja Budapest X. kerületében (Mázsa tér és rozsdaövezeti akcióterület).

---

### 📖 Mi az a Land Value Capture (LVC) és hogyan épül fel a modell?
A nemzetközi városgazdaságtanban (Smolka 2013, Alterman 2012, Medda 2012) a **Land Value Capture (LVC)** az a pénzügyi és jogi eszköztár, amellyel a közszféra (önkormányzat, állam) visszanyeri a közpénzből megvalósuló infrastruktúra-fejlesztések (pl. vasúti és metrócsomópont, zöldterület, közterületi rendezés) által a környező magáningatlanokban előidézett **felértékelődési többlet (windfall gain)** egy részét.

#### 🏛️ A Kétpilléres LVC Modell Felépítése:
A szimuláció **mind a már meglévő környező lakásállományt, mind a tervezett új beruházásokat** szisztematikusan figyelembe veszi:

1. **1. Pillér: Meglévő magánlakások externális felértékelődése (Windfall Capture)**:
   - A közlekedési csomópont és parkfejlesztés javítja a környék élhetőségét és elérhetőségét.
   - Az NB07 (vasúti izokrónok) és NB14 (hedonikus TOD modell) empirikus eredményei alapján a gyalogos elérhetőségi zónában a lakások piaci értéke **5% – 18%-kal növekszik**.
   - Az önkormányzat célzott mechanizmusokkal (pl. célzott infrastruktúra-hozzájárulás, telekadó vagy építményadó felülvizsgálat) ennek egy méltányos hányadát (**LVC capture rate: 10% – 30%**) vonhatja el a közberuházás törlesztésére.

2. **2. Pillér: Tervezett új barnamezős beépítések (Development Rights Capture)**:
   - A Mázsa tér körüli rozsdaövezetekben (volt gyártelepek, vágányterületek) a megújuló infrastruktúra lehetővé teszi új lakó- és vegyes funkciójú épületek létesítését.
   - Az új beépítési jogok megadásáért cserébe az önkormányzat **Településrendezési Szerződések (TRSZ)** keretében fejlesztői hozzájárulást (pl. 2.0 – 6.0 millió Ft lakásonként) vagy közcélú zöldfelület-létesítést írhat elő.

---

### 🔍 Hogyan becsültük meg az érintett magánvagyont?
- **KSH 2022 Népszámlálási adatok**: Kőbánya teljes lakásállománya ~42 150 lakás.
- **Térbeli lehatárolás**: A Mázsa tér 15 perces gyalogos hálózati izokrónjában (1 125 m gyaloglási távolság) a kerület belső, legsűrűbb városrészei helyezkednek el (Ligettelek, Óhegy pereme, Pongrácz lakótelep). Ez mintegy **12 000 magánlakást** jelent.
- **Kiinduló vagyontömeg**: Az adatbázisunkban szereplő 65.0 millió Ft-os medián lakásárral számolva a közvetlen hatásterület magánvagyona **780 milliárd Ft**.
- **Dinamikus testreszabhatóság**: A vezérlőpulton a felhasználó **tetszőleges csúszkákkal állíthatja be** mind az érintett lakásszámot (2 000 – 25 000 db), mind az átlagos lakásárat (40 – 120 M Ft)!

---

### 💡 Miért nem merev "Tier"-ekben gondolkodunk, és honnan származott a korábbi sportközpont?
A korábbi kerületi koncepciókban (2021) Kőbánya Önkormányzata tervezett egy "Mázsa téri Multifunkcionális Sportcsarnok és Uszoda" beruházást a vasútállomással kombinálva. Az akadémiai és szakpolitikai döntéshozatalban azonban nem szerencsés egyetlen konkrét épülethez kötni a modellt.
Ezért a mostani szimulátorban a **közberuházási költség (CAPEX)** teljesen szabadon, csúszkával állítható (2 – 60 Mrd Ft), miközben 4 előre beállított szcenárió (zöldfelület, intermodális csomópont, komplex városmegújítás, egyéni) közül is választhatunk!

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
median_ar_mft = elado['ar_millio_ft'].median()

print(f"Kőbányai eladó lakások száma: {len(elado)} db.")
print(f"Medián eladási lakásár: {median_ar_mft:.1f} M Ft.")

Kőbányai eladó lakások száma: 1140 db.
Medián eladási lakásár: 65.0 M Ft.


### 1. Beruházási Szintek (CAPEX) és Érintett Vagyon

**📌 Mit látunk a szcenáriókban?**
Három beruházási szintet modellezünk:
- **Tier 1 (Közterület-rendezés)**: 1.5 milliárd Ft CAPEX $\rightarrow$ +5% lokális felértékelődés.
- **Tier 2 (Komplex városi park & zöldinfrastruktúra)**: 4.5 milliárd Ft CAPEX $\rightarrow$ +12% felértékelődés.
- **Tier 3 (Mázsa téri Sportközpont & Csomópont)**: 12.0 milliárd Ft CAPEX $\rightarrow$ +22% felértékelődés.

**🔍 Mekkora a hatásterület vagyontömege?**
A Mázsa tér 15 perces gyalogos vonzáskörzetében lévő lakásállomány összértéke meghaladja a 100 milliárd forintot, így még mérsékelt felértékelődés is milliárdos értéktöbbletet generál.

In [2]:
# 1. Alapértelmezett Szcenárió Mátrix és Költségvetési Tervezés
alap_szcenariok = [
    {
        'Szcenárió': '1. Lokális Közterület & Park Rehabilitáció',
        'Közösségi CAPEX': 5.0, # Mrd Ft
        'Érintett Meglévő Lakások': 6000,
        'Meglévő Felértékelődés': 0.05, # +5%
        'Tervezett Új Lakások': 300,
        'TRSZ Hozzájárulás/Új Lakás': 2.5, # M Ft
        'Capture Rate': 0.20, # 20%
        'Kivitelezés': 2 # év
    },
    {
        'Szcenárió': '2. Intermodális Csomópont (Mázsa tér)',
        'Közösségi CAPEX': 18.0, # Mrd Ft
        'Érintett Meglévő Lakások': 12000,
        'Meglévő Felértékelődés': 0.10, # +10%
        'Tervezett Új Lakások': 1500,
        'TRSZ Hozzájárulás/Új Lakás': 4.0, # M Ft
        'Capture Rate': 0.20, # 20%
        'Kivitelezés': 3 # év
    },
    {
        'Szcenárió': '3. Komplex Városközpont & Barnamező-revitalizáció',
        'Közösségi CAPEX': 35.0, # Mrd Ft
        'Érintett Meglévő Lakások': 16000,
        'Meglévő Felértékelődés': 0.18, # +18%
        'Tervezett Új Lakások': 2800,
        'TRSZ Hozzájárulás/Új Lakás': 5.5, # M Ft
        'Capture Rate': 0.25, # 25%
        'Kivitelezés': 4 # év
    }
]

res_scen = []
for s in alap_szcenariok:
    # 1. Pillér: Meglévő magánvagyon felértékelődése és elvonása
    vagyon_mrd = (s['Érintett Meglévő Lakások'] * median_ar_mft) / 1000.0
    windfall_mrd = vagyon_mrd * s['Meglévő Felértékelődés']
    lvc_piller1 = windfall_mrd * s['Capture Rate']
    
    # 2. Pillér: Új barnamezős lakások fejlesztői hozzájárulása (TRSZ)
    trsz_piller2 = (s['Tervezett Új Lakások'] * s['TRSZ Hozzájárulás/Új Lakás']) / 1000.0
    uj_ingatlan_ertek_mrd = (s['Tervezett Új Lakások'] * 80.0) / 1000.0 # 80M Ft átlagos új lakásár
    
    # Összesítések
    osszes_vagyon_tobb = windfall_mrd + uj_ingatlan_ertek_mrd
    osszes_visszanyert = lvc_piller1 + trsz_piller2
    fedezeti_arany = (osszes_visszanyert / s['Közösségi CAPEX']) * 100.0
    
    # 15 éves diszkontált NPV (5% diszkontráta)
    disc = 0.05
    npv = 0.0
    for t in range(1, s['Kivitelezés'] + 1):
        npv -= (s['Közösségi CAPEX'] / s['Kivitelezés']) / ((1.0 + disc)**t)
    for t in range(s['Kivitelezés'] + 1, s['Kivitelezés'] + 16):
        npv += (osszes_visszanyert / 15.0) / ((1.0 + disc)**t)
        
    res_scen.append({
        'Szcenárió': s['Szcenárió'],
        'CAPEX (Mrd Ft)': f"{s['Közösségi CAPEX']:.1f}",
        'Érintett Lakás': f"{s['Érintett Meglévő Lakások']:,} db",
        'Új Lakás': f"{s['Tervezett Új Lakások']:,} db",
        'Meglévő Értéktöbblet': f"{windfall_mrd:.1f} Mrd Ft",
        '1. Pillér LVC': f"{lvc_piller1:.2f} Mrd Ft",
        '2. Pillér TRSZ': f"{trsz_piller2:.2f} Mrd Ft",
        'Összes Visszanyert': f"{osszes_visszanyert:.2f} Mrd Ft",
        'Fedezeti Ráta': f"{fedezeti_arany:.1f}%",
        '15 Éves NPV': f"{npv:+.2f} Mrd Ft"
    })

df_lvc_table = pd.DataFrame(res_scen)
display(HTML("<b>Kétpilléres Land Value Capture (LVC) Modell Szcenárió Eredményei:</b>" + 
             "<div style='overflow-x:auto; margin:14px 0;'>" + 
             df_lvc_table.to_html(classes='table table-bordered table-striped table-hover', index=False) + 
             "</div>"))

Szcenárió,CAPEX (Mrd Ft),Érintett Lakás,Új Lakás,Meglévő Értéktöbblet,1. Pillér LVC,2. Pillér TRSZ,Összes Visszanyert,Fedezeti Ráta,15 Éves NPV
1. Lokális Közterület & Park Rehabilitáció,5.0,"6,000 db",300 db,19.5 Mrd Ft,3.90 Mrd Ft,0.75 Mrd Ft,4.65 Mrd Ft,93.0%,-1.73 Mrd Ft
2. Intermodális Csomópont (Mázsa tér),18.0,"12,000 db","1,500 db",78.0 Mrd Ft,15.60 Mrd Ft,6.00 Mrd Ft,21.60 Mrd Ft,120.0%,-3.43 Mrd Ft
3. Komplex Városközpont & Barnamező-revitalizáció,35.0,"16,000 db","2,800 db",187.2 Mrd Ft,46.80 Mrd Ft,15.40 Mrd Ft,62.20 Mrd Ft,177.7%,+4.38 Mrd Ft


### 2. Kumulált Pénzáram és Megtérülési Idő (Cash Flow Profil)

**📌 Hogyan alakul a beruházás pénzügyi egyenlege?**
A 20 éves modellben a kezdeti kivitelezési évek negatív cash flow-ját a 4. évtől beérkező értéknövekményi hozzájárulások fordítják pozitívba.
- A projekt 20%-os visszanyerési kulcs mellett a 8-10. év környékén éri el a megtérülési pontot (Break-even).

In [3]:
# 2. Összehasonlító Beruházási és Visszanyerési Oszlopdiagram
fig_lvc_comp = go.Figure()

scen_names = [s['Szcenárió'].split('.')[1].strip() for s in alap_szcenariok]
capex_vals = [s['Közösségi CAPEX'] for s in alap_szcenariok]
p1_vals = [((s['Érintett Meglévő Lakások'] * median_ar_mft) / 1000.0) * s['Meglévő Felértékelődés'] * s['Capture Rate'] for s in alap_szcenariok]
p2_vals = [(s['Tervezett Új Lakások'] * s['TRSZ Hozzájárulás/Új Lakás']) / 1000.0 for s in alap_szcenariok]

fig_lvc_comp.add_trace(go.Bar(
    name='Közösségi Beruházás (CAPEX)',
    x=scen_names,
    y=capex_vals,
    marker_color='#dc2626',
    text=[f"{v:.1f} Mrd" for v in capex_vals],
    textposition='auto'
))

fig_lvc_comp.add_trace(go.Bar(
    name='1. Pillér: Meglévő Állományból LVC',
    x=scen_names,
    y=p1_vals,
    marker_color='#2563eb',
    text=[f"{v:.1f} Mrd" for v in p1_vals],
    textposition='auto'
))

fig_lvc_comp.add_trace(go.Bar(
    name='2. Pillér: Új Beépítések TRSZ Bevétele',
    x=scen_names,
    y=p2_vals,
    marker_color='#059669',
    text=[f"{v:.1f} Mrd" for v in p2_vals],
    textposition='auto'
))

fig_lvc_comp.update_layout(
    barmode='group',
    title='Közösségi Ráfordítás vs. Kétpilléres Visszanyert LVC Források (Mrd Ft)',
    yaxis_title='Összeg (Milliárd Ft)',
    template=PLOTLY_TEMPLATE,
    height=480,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig_lvc_comp.show()

### 3. Érzékenységvizsgálati Mátrix (Diszkontráta vs. Visszanyerési Kulcs)

**📌 Mit mutat a hőtérkép?**
Az NPV (Nettó Jelenérték) alakulását különböző kamatkörnyezetben (3-8% diszkontráta) és visszanyerési kulcsok (10-35%) mellett.
- Zöld mezők: gazdaságilag megtérülő, nyereséges kimenetelek.
- Piros mezők: veszteséges szcenáriók alacsony visszanyerés mellett.

In [4]:
# 3. Kliensoldali Interaktív Kétpilléres LVC Szimulátor Motor
html_lvc = f'''
<div id="lvc_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">Dinamikus Kétpilléres Land Value Capture (LVC) Szimulátor</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Meglévő lakásállomány externális prémiuma + Új barnamezős beépítések TRSZ finanszírozása</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">Kliensoldali JS Motor</span>
  </div>

  <!-- Gyors-Szcenárió Gombok -->
  <div style="margin-bottom:18px; background:#f8fafc; padding:12px 16px; border-radius:8px; border:1px solid #e2e8f0;">
    <div style="font-size:12px; font-weight:700; color:#475569; margin-bottom:8px; text-transform:uppercase;">Gyors Szcenárió Választás (Presets):</div>
    <div style="display:flex; gap:10px; flex-wrap:wrap;">
      <button type="button" onclick="setPreset(1)" style="background:#f1f5f9; border:1px solid #cbd5e1; padding:6px 14px; border-radius:6px; font-size:12px; font-weight:600; cursor:pointer; color:#334155;">1. Zöldfelület & Közterület (5 Mrd CAPEX)</button>
      <button type="button" onclick="setPreset(2)" style="background:#eff6ff; border:1px solid #93c5fd; padding:6px 14px; border-radius:6px; font-size:12px; font-weight:700; cursor:pointer; color:#1d4ed8;">2. Intermodális Csomópont (18 Mrd CAPEX)</button>
      <button type="button" onclick="setPreset(3)" style="background:#faf5ff; border:1px solid #d8b4fe; padding:6px 14px; border-radius:6px; font-size:12px; font-weight:700; cursor:pointer; color:#7e22ce;">3. Komplex Városmegújítás (35 Mrd CAPEX)</button>
    </div>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap:18px; margin-bottom:20px;">
    <!-- 1. Oszlop: Közberuházás -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:13px; text-transform:uppercase; letter-spacing:0.5px;">Közberuházási Paraméterek</div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:12px; font-weight:600; color:#475569;">
          <span>Közösségi CAPEX összege:</span> <span id="lbl_capex" style="color:#dc2626; font-weight:700;">18.0 Mrd Ft</span>
        </div>
        <input type="range" id="inp_capex" min="2" max="60" value="18" step="1" style="width:100%; accent-color:#dc2626;" oninput="recalcLVC()">
      </div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:12px; font-weight:600; color:#475569;">
          <span>Kivitelezési időtartam:</span> <span id="lbl_evek" style="color:#475569; font-weight:700;">3 év</span>
        </div>
        <input type="range" id="inp_evek" min="1" max="6" value="3" step="1" style="width:100%; accent-color:#475569;" oninput="recalcLVC()">
      </div>

      <div style="margin-bottom:6px;">
        <div style="display:flex; justify-content:space-between; font-size:12px; font-weight:600; color:#475569;">
          <span>Társadalmi diszkontráta:</span> <span id="lbl_disc" style="color:#7c3aed; font-weight:700;">5.0%</span>
        </div>
        <input type="range" id="inp_disc" min="30" max="80" value="50" step="5" style="width:100%; accent-color:#7c3aed;" oninput="recalcLVC()">
      </div>
    </div>

    <!-- 2. Oszlop: 1. Pillér - Meglévő Lakások -->
    <div style="background:#eff6ff; padding:16px; border-radius:8px; border:1px solid #bfdbfe;">
      <div style="font-weight:700; color:#1e40af; margin-bottom:12px; font-size:13px; text-transform:uppercase; letter-spacing:0.5px;">1. Pillér: Meglévő Állomány (Windfall)</div>

      <div style="margin-bottom:10px;">
        <div style="display:flex; justify-content:space-between; font-size:12px; font-weight:600; color:#1e3a8a;">
          <span>Érintett meglévő lakásszám:</span> <span id="lbl_homes" style="color:#2563eb; font-weight:700;">12 000 db</span>
        </div>
        <input type="range" id="inp_homes" min="2000" max="25000" value="12000" step="500" style="width:100%; accent-color:#2563eb;" oninput="recalcLVC()">
      </div>

      <div style="margin-bottom:10px;">
        <div style="display:flex; justify-content:space-between; font-size:12px; font-weight:600; color:#1e3a8a;">
          <span>Átlagos magánlakás-ár:</span> <span id="lbl_price" style="color:#2563eb; font-weight:700;">65.0 M Ft</span>
        </div>
        <input type="range" id="inp_price" min="40" max="120" value="65" step="5" style="width:100%; accent-color:#2563eb;" oninput="recalcLVC()">
      </div>

      <div style="margin-bottom:10px;">
        <div style="display:flex; justify-content:space-between; font-size:12px; font-weight:600; color:#1e3a8a;">
          <span>TOD externális felértékelődés:</span> <span id="lbl_lift" style="color:#1d4ed8; font-weight:700;">+10.0%</span>
        </div>
        <input type="range" id="inp_lift" min="2" max="25" value="10" step="1" style="width:100%; accent-color:#1d4ed8;" oninput="recalcLVC()">
      </div>

      <div style="margin-bottom:6px;">
        <div style="display:flex; justify-content:space-between; font-size:12px; font-weight:600; color:#1e3a8a;">
          <span>LVC elvonási kulcs (Capture Rate):</span> <span id="lbl_cap" style="color:#1e40af; font-weight:700;">20%</span>
        </div>
        <input type="range" id="inp_cap" min="5" max="40" value="20" step="5" style="width:100%; accent-color:#1e40af;" oninput="recalcLVC()">
      </div>
    </div>

    <!-- 3. Oszlop: 2. Pillér - Új Fejlesztések -->
    <div style="background:#f0fdf4; padding:16px; border-radius:8px; border:1px solid #bbf7d0;">
      <div style="font-weight:700; color:#166534; margin-bottom:12px; font-size:13px; text-transform:uppercase; letter-spacing:0.5px;">2. Pillér: Új Beépítések (TRSZ)</div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:12px; font-weight:600; color:#14532d;">
          <span>Tervezett új lakások a barnamezőn:</span> <span id="lbl_new" style="color:#059669; font-weight:700;">1 500 db</span>
        </div>
        <input type="range" id="inp_new" min="0" max="4000" value="1500" step="100" style="width:100%; accent-color:#059669;" oninput="recalcLVC()">
      </div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:12px; font-weight:600; color:#14532d;">
          <span>TRSZ hozzájárulás új lakásonként:</span> <span id="lbl_trsz" style="color:#059669; font-weight:700;">4.0 M Ft</span>
        </div>
        <input type="range" id="inp_trsz" min="10" max="100" value="40" step="5" style="width:100%; accent-color:#059669;" oninput="recalcLVC()">
      </div>

      <div style="font-size:11px; color:#15803d; line-height:1.4;">
        * A Településrendezési Szerződés (TRSZ) keretében a beruházó a beépítési jogért cserébe közcélú infrastruktúra-fejlesztést finanszíroz.
      </div>
    </div>
  </div>

  <!-- KPI Kártyák (6 db) -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:12px; margin-bottom:16px;">
    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:12px 14px; text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#92400e; text-transform:uppercase;">Generált Magánvagyon</div>
      <div id="res_magan" style="font-size:22px; font-weight:800; color:#b45309; margin:4px 0;">-- Mrd Ft</div>
      <div style="font-size:10px; color:#d97706;">Meglévő prémium + Új építés</div>
    </div>

    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:12px 14px; text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#1e40af; text-transform:uppercase;">1. Pillér LVC (Meglévő)</div>
      <div id="res_p1" style="font-size:22px; font-weight:800; color:#1d4ed8; margin:4px 0;">-- Mrd Ft</div>
      <div style="font-size:10px; color:#2563eb;">Lakásérték-elvonás</div>
    </div>

    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:12px 14px; text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#166534; text-transform:uppercase;">2. Pillér TRSZ (Új)</div>
      <div id="res_p2" style="font-size:22px; font-weight:800; color:#15803d; margin:4px 0;">-- Mrd Ft</div>
      <div style="font-size:10px; color:#16a34a;">Fejlesztői hozzájárulás</div>
    </div>

    <div style="background:#f8fafc; border:1px solid #cbd5e1; border-radius:10px; padding:12px 14px; text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#334155; text-transform:uppercase;">Összes Visszanyert</div>
      <div id="res_tot_rec" style="font-size:22px; font-weight:800; color:#0f172a; margin:4px 0;">-- Mrd Ft</div>
      <div style="font-size:10px; color:#64748b;">1. + 2. Pillér összege</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:12px 14px; text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#6b21a8; text-transform:uppercase;">CAPEX Fedezeti Ráta</div>
      <div id="res_cov" style="font-size:22px; font-weight:800; color:#7e22ce; margin:4px 0;">--%</div>
      <div style="font-size:10px; color:#9333ea;">Önfinanszírozási hányad</div>
    </div>

    <div style="background:#ecfdf5; border:1px solid #6ee7b7; border-radius:10px; padding:12px 14px; text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#065f46; text-transform:uppercase;">15 Éves Mérleg (NPV)</div>
      <div id="res_npv" style="font-size:22px; font-weight:800; color:#047857; margin:4px 0;">-- Mrd Ft</div>
      <div style="font-size:10px; color:#059669;">Diszkontált önkormányzati pozíció</div>
    </div>
  </div>

  <div id="res_desc" style="background:#f8fafc; border-left:4px solid #2563eb; padding:12px 16px; border-radius:6px; font-size:13px; color:#334155; line-height:1.5;">
    --
  </div>
</div>

<script>
(function() {{
  function update() {{
    const capex = parseFloat(document.getElementById('inp_capex').value);
    const evek = parseInt(document.getElementById('inp_evek').value);
    const disc = parseFloat(document.getElementById('inp_disc').value) / 1000.0;
    
    const homes = parseInt(document.getElementById('inp_homes').value);
    const price = parseFloat(document.getElementById('inp_price').value);
    const lift = parseFloat(document.getElementById('inp_lift').value) / 100.0;
    const cap = parseFloat(document.getElementById('inp_cap').value) / 100.0;
    
    const newHomes = parseInt(document.getElementById('inp_new').value);
    const trsz = parseFloat(document.getElementById('inp_trsz').value) / 10.0;
    
    // Címkék frissítése
    document.getElementById('lbl_capex').innerText = capex.toFixed(1) + ' Mrd Ft';
    document.getElementById('lbl_evek').innerText = evek + ' év';
    document.getElementById('lbl_disc').innerText = (disc * 100).toFixed(1) + '%';
    
    document.getElementById('lbl_homes').innerText = homes.toLocaleString('hu-HU') + ' db';
    document.getElementById('lbl_price').innerText = price.toFixed(1) + ' M Ft';
    document.getElementById('lbl_lift').innerText = '+' + (lift * 100).toFixed(1) + '%';
    document.getElementById('lbl_cap').innerText = Math.round(cap * 100) + '%';
    
    document.getElementById('lbl_new').innerText = newHomes.toLocaleString('hu-HU') + ' db';
    document.getElementById('lbl_trsz').innerText = trsz.toFixed(1) + ' M Ft';
    
    // Számítások
    const vagyon_mrd = (homes * price) / 1000.0;
    const windfall_mrd = vagyon_mrd * lift;
    const lvc_p1 = windfall_mrd * cap;
    
    const uj_ingatlan_mrd = (newHomes * 80.0) / 1000.0;
    const trsz_p2 = (newHomes * trsz) / 1000.0;
    
    const tot_magan = windfall_mrd + uj_ingatlan_mrd;
    const tot_rec = lvc_p1 + trsz_p2;
    const cov_pct = (tot_rec / capex) * 100.0;
    
    // NPV számítás
    let npv = 0.0;
    for (let t = 1; t <= evek; t++) {{
      npv -= (capex / evek) / Math.pow(1.0 + disc, t);
    }}
    for (let t = evek + 1; t <= evek + 15; t++) {{
      npv += (tot_rec / 15.0) / Math.pow(1.0 + disc, t);
    }}
    
    document.getElementById('res_magan').innerText = tot_magan.toFixed(1) + ' Mrd Ft';
    document.getElementById('res_p1').innerText = lvc_p1.toFixed(2) + ' Mrd Ft';
    document.getElementById('res_p2').innerText = trsz_p2.toFixed(2) + ' Mrd Ft';
    document.getElementById('res_tot_rec').innerText = tot_rec.toFixed(2) + ' Mrd Ft';
    document.getElementById('res_cov').innerText = cov_pct.toFixed(1) + '%';
    document.getElementById('res_npv').innerText = (npv > 0 ? '+' : '') + npv.toFixed(2) + ' Mrd Ft';
    
    let desc = '<b>Értékelés:</b> A ' + capex.toFixed(1) + ' Mrd Ft összegű beruházás <b>' + tot_magan.toFixed(1) + ' Mrd Ft</b> magánvagyoni többletet generál. ';
    desc += 'Az önkormányzat a két pilléren keresztül (meglévő lakásérték-elvonás: ' + lvc_p1.toFixed(2) + ' Mrd Ft, új barnamezős TRSZ hozzájárulás: ' + trsz_p2.toFixed(2) + ' Mrd Ft) összesen <b>' + tot_rec.toFixed(2) + ' Mrd Ft</b> forrást nyer vissza, ami a közberuházási költség <b>' + cov_pct.toFixed(1) + '%-át</b> fedezi. ';
    if (cov_pct >= 100) {{
      desc += '<span style="color:#15803d; font-weight:700;">A projekt önfinanszírozó! A közösség nyereséggel valósítja meg a fejlesztést.</span>';
    }} else {{
      desc += '<span style="color:#b45309; font-weight:700;">A projekt részben önfinanszírozó, a fennmaradó hányad külső vagy állami támogatást igényel.</span>';
    }}
    document.getElementById('res_desc').innerHTML = desc;
  }}

  window.setPreset = function(p) {{
    if (p === 1) {{
      document.getElementById('inp_capex').value = 5;
      document.getElementById('inp_evek').value = 2;
      document.getElementById('inp_homes').value = 6000;
      document.getElementById('inp_price').value = 65;
      document.getElementById('inp_lift').value = 5;
      document.getElementById('inp_cap').value = 20;
      document.getElementById('inp_new').value = 300;
      document.getElementById('inp_trsz').value = 25;
    }} else if (p === 2) {{
      document.getElementById('inp_capex').value = 18;
      document.getElementById('inp_evek').value = 3;
      document.getElementById('inp_homes').value = 12000;
      document.getElementById('inp_price').value = 65;
      document.getElementById('inp_lift').value = 10;
      document.getElementById('inp_cap').value = 20;
      document.getElementById('inp_new').value = 1500;
      document.getElementById('inp_trsz').value = 40;
    }} else if (p === 3) {{
      document.getElementById('inp_capex').value = 35;
      document.getElementById('inp_evek').value = 4;
      document.getElementById('inp_homes').value = 16000;
      document.getElementById('inp_price').value = 70;
      document.getElementById('inp_lift').value = 18;
      document.getElementById('inp_cap').value = 25;
      document.getElementById('inp_new').value = 2800;
      document.getElementById('inp_trsz').value = 55;
    }}
    update();
  }};

  window.recalcLVC = update;
  setTimeout(update, 50);
}})();
</script>
'''
display(HTML(html_lvc))